In [ ]:
import numpy as np
import anndata as ad
from matplotlib import pyplot as plt
import os
from sklearn.metrics import roc_auc_score

from scunveil import scUnveil

from constants import DATA_ROOT, SC_UNVEIL_VERSION, ANALYSIS_ROOT

In [ ]:
ADATA = ad.read_h5ad(DATA_ROOT / "downstream_datasets/Hao_PBMC/Hao_PBMC.h5ad")

In [ ]:
MAGIC_RNA = np.load(DATA_ROOT / "downstream_datasets/Hao_PBMC/imputation/magic.npy")

In [ ]:
SC_UNVEIL = scUnveil(model_version=SC_UNVEIL_VERSION)
SC_UNVEIL_RESULT = SC_UNVEIL.process_anndata(ADATA, batch_size=1000)

In [ ]:


def auroc(a, b):
    a, b = np.asarray(a).ravel(), np.asarray(b).ravel()

    if not a.size or not b.size:
        raise ValueError("Both arrays must be non-empty.")

    values = np.r_[a, b]
    if not np.isfinite(values).all():
        raise ValueError("Inputs must contain only finite values.")

    labels = np.r_[np.zeros(a.size), np.ones(b.size)]
    return roc_auc_score(labels, values)

def best_auroc(a, b):
    score = auroc(a, b)
    return max(score, 1 - score)


def compare_imputation(gene_name, cell_types_compare, cell_type_level, batch_size=1000):
    gene_index = list(ADATA.raw.var['feature_name']).index(gene_name)

    raw_rna = np.array(ADATA.raw.X[:, gene_index].todense()).flatten()
    magic_impute = MAGIC_RNA[:, gene_index]
    sc_unveil_impute = SC_UNVEIL_RESULT.get_specific_genes_imputation([gene_name], batch_size=batch_size).X.flatten()
    

    cell_type_array = np.array(list(ADATA.obs[cell_type_level]))
    
    plt.figure(figsize=(9, 2.5))

    # RNA   
    array = raw_rna
    sep_arrays = []
    plt.subplot(1, 3, 1)
    for cell_type in cell_types_compare:
        if cell_type is None:
            select = np.ones_like(cell_type_array).astype('bool')            
        else:
            select = cell_type_array == cell_type

        sep_arrays.append(array[select].copy())

        bar_x, bar_y = np.unique(array[select], return_counts=True)
        bar_y = bar_y / bar_y.sum()
        plt.bar(bar_x, bar_y, alpha=0.5, label=cell_type)

    separ = best_auroc(sep_arrays[0], sep_arrays[1])

    plt.grid()
    plt.legend()
    plt.title(f'AUROC: {100*separ:.1f} %')
    plt.xlabel(f'#UMIs {gene_name}')
    plt.ylabel('Density')


    # MAGIC   
    array = magic_impute
    sep_arrays = []
    plt.subplot(1, 3, 2)
    for cell_type in cell_types_compare:
        if cell_type is None:
            select = np.ones_like(cell_type_array).astype('bool')            
        else:
            select = cell_type_array == cell_type

        sep_arrays.append(array[select].copy())

        plt.hist(array[select], 50, density=True, alpha=0.5, label=cell_type)

    separ = best_auroc(sep_arrays[0], sep_arrays[1])

    plt.grid()
    plt.legend()
    plt.title(f'AUROC: {100*separ:.1f} %')
    plt.xlabel(f'MAGIC Impute {gene_name}')
    plt.ylabel('Density')

    # scUNVEIL   
    array = sc_unveil_impute
    sep_arrays = []
    plt.subplot(1, 3, 3)
    for cell_type in cell_types_compare:
        if cell_type is None:
            select = np.ones_like(cell_type_array).astype('bool')            
        else:
            select = cell_type_array == cell_type

        sep_arrays.append(array[select].copy())

        plt.hist(array[select], 50, density=True, alpha=0.5, label=cell_type)

    separ = best_auroc(sep_arrays[0], sep_arrays[1])  

    plt.grid()
    plt.legend()
    plt.title(f'AUROC: {100*separ:.1f} %')
    plt.xlabel(f'scUNVEIL Impute {gene_name}')
    plt.ylabel('Density')

    os.makedirs(ANALYSIS_ROOT / 'Hao_PBMC/figures', exist_ok=True)
    plt.tight_layout()
    plt.savefig(
        ANALYSIS_ROOT / (
            f'Hao_PBMC/figures/'
            f'{gene_name}_{cell_types_compare[0]}-{cell_types_compare[1]}_'
            f'{cell_type_level}.svg'
        ),
        bbox_inches='tight',
        # pad_inches=0.1,
    )
plt.show()

In [ ]:
compare_imputation('CD4', ['CD8 T', 'CD4 T'], 'celltype.l1')

In [ ]:
compare_imputation('FOXP3', ['CD4 TCM', 'Treg'], 'celltype.l2')